In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from pathlib import Path

from config import YELLOW_CLEAN_PARQUET, HVFHV_CLEAN_PARQUET, UBER_LICENSE, LYFT_LICENSE

In [2]:
yellow = pq.read_table(YELLOW_CLEAN_PARQUET).to_pandas()
hvfhv = pq.read_table(HVFHV_CLEAN_PARQUET).to_pandas()

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

def table_day_share(df, dataset_name):

    summary = (
        df["day_name"]
        .value_counts(normalize=True)
        .reindex(day_order)
        .mul(100)
        .round(2)
        .reset_index()
    )

    summary.columns = ["day_name", "trip_share_pct"]
    summary["dataset"] = dataset_name

    return summary[
        ["dataset", "day_name", "trip_share_pct"]
    ]


hvfhv_day_table = table_day_share(
    hvfhv,
    "hvfhv Uber/Lyft"
)

yellow_day_table = table_day_share(
    yellow,
    "Yellow Taxi"
)

display(hvfhv_day_table)
display(yellow_day_table)

In [ ]:
def table_hour_share(df, dataset_name):
    
    summary = (
        df["pickup_hour"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
        .reset_index()
    )

    summary.columns = ["pickup_hour", "trip_share_pct"]
    summary["dataset"] = dataset_name

    return summary[
        ["dataset", "pickup_hour", "trip_share_pct"]
    ]

display(table_hour_share(hvfhv, "hvfhv"))
display(table_hour_share(yellow, "Yellow"))

In [ ]:
borough_order = [
    "Manhattan",
    "Brooklyn",
    "Queens",
    "Bronx",
    "Staten Island",
    "Unknown"
]

def table_borough_share(df, dataset_name):

    summary = (
        df["PU_Borough"]
        .value_counts(normalize=True)
        .reindex(borough_order)
        .mul(100)
        .round(2)
        .reset_index()
    )

    summary.columns = ["borough", "trip_share_pct"]
    summary["dataset"] = dataset_name

    return summary[
        ["dataset", "borough", "trip_share_pct"]
    ]

display(table_borough_share(hvfhv, "hvfhv"))
display(table_borough_share(yellow, "Yellow"))

In [ ]:
weather_summary = (
    merged_df
    .groupby("precipitation_bucket")
    .agg(
        trips=("trip_id", "count"),
        avg_fare=("fare_amount", "mean"),
        avg_duration=("trip_duration_min", "mean")
    )
    .round(2)
    .reset_index()
)

display(weather_summary)

In [ ]:
def classify_peak(hour):
    
    if hour in [7,8,9,16,17,18]:
        return "Peak"
    else:
        return "Off-Peak"

hvfhv["peak_type"] = hvfhv["pickup_hour"].apply(classify_peak)

peak_summary = (
    hvfhv
    .groupby("peak_type")
    .agg(
        trips=("hvfhs_license_num", "count"),
        avg_fare=("base_passenger_fare", "mean"),
        avg_driver_pay=("driver_pay", "mean")
    )
    .round(2)
    .reset_index()
)

display(peak_summary)

### Distribution plots

In [ ]:
cols_yellow = [
    "trip_miles", "trip_duration_min", "fare", "total_amount", 
    "tips", "extra", "tolls", "congestion_surcharge", "cbd_congestion_fee"
]

cols_hvfhv = [
    "trip_miles", "trip_duration_min", "fare", "driver_pay",
    "tips", "congestion_surcharge", "cbd_congestion_fee", 
]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_distributions(df, cols, title):
    n = len(cols)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))

    for i, col in enumerate(cols):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i])
        axes[i].set_title(f"{col} distribution")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_distributions(yellow, cols_yellow, "Yellow Taxi Distributions")

In [ ]:
plot_distributions(hvfhv, cols_hvfhv, "hvfhv (Uber + Lyft) Distributions")

fare → skewed right → heavy tail (surge / long trips)
distance → short trips dominate
duration → similar pattern
passenger_count → mostly 1–2

short trips + high variability → users care about ETA more than price

In [ ]:
sns.pairplot(
    yellow[cols_yellow].sample(5000),
    diag_kind="kde"
)
plt.suptitle("Yellow Taxi Pairplot", y=1.02)
plt.show()

In [ ]:
sns.pairplot(
    hvfhv[cols_hvfhv].sample(5000),
    diag_kind="kde"
)
plt.suptitle("hvfhv Pairplot", y=1.02)
plt.show()

## EDA 1 — Trip distribution by time

In [ ]:
## Trip proportion by time zone
def plot_share(series, title):
    share = series.value_counts(normalize=True).sort_index() * 100
    ax = share.plot(kind="bar", figsize=(8,4))
    ax.set_title(title)
    ax.set_ylabel("Share of trips (%)")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.show()
    display(share.round(2).rename("trip_share_pct"))

plot_share(hvfhv["time_zone"], "HVFHV Uber/Lyft Trip Share by Time Zone")
plot_share(yellow["time_zone"], "Yellow Taxi Trip Share by Time Zone")

In [ ]:
## Trip proportion by weekday / weekend
plot_share(hvfhv["week_category"], "hvfhv Uber/Lyft Trip Share: Weekday vs Weekend")
plot_share(yellow["week_category"], "Yellow Taxi Trip Share: Weekday vs Weekend")

In [ ]:
## Trip distribution by day name
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def plot_day_share(df, title):
    share = df["day_name"].value_counts(normalize=True).reindex(day_order) * 100
    ax = share.plot(kind="bar", figsize=(9,4))
    ax.set_title(title)
    ax.set_ylabel("Share of trips (%)")
    ax.set_xlabel("")
    plt.xticks(rotation=30)
    plt.show()
    display(share.round(2).rename("trip_share_pct"))

plot_day_share(hvfhv, "hvfhv Uber/Lyft Trip Share by Day")
plot_day_share(yellow, "Yellow Taxi Trip Share by Day")

## EDA 2 — Daily / monthly time series

In [ ]:
## Uber vs Lyft daily ride volume
daily_hvfhv = (
    hvfhv
    .groupby(["date", "company"])
    .size()
    .reset_index(name="rides")
)

daily_pivot = daily_hvfhv.pivot(index="date", columns="company", values="rides").fillna(0)

ax = daily_pivot.plot(figsize=(14,5))
ax.set_title("Daily Ride Volume: Uber vs Lyft")
ax.set_ylabel("Number of rides")
ax.set_xlabel("")
plt.show()

display(daily_pivot.head())

In [ ]:
## Monthly ride volume
monthly_hvfhv = (
    hvfhv
    .groupby(["month", "company"])
    .size()
    .reset_index(name="rides")
)

monthly_pivot = monthly_hvfhv.pivot(index="month", columns="company", values="rides").fillna(0)

ax = monthly_pivot.plot(kind="line", marker="o", figsize=(14,5))
ax.set_title("Monthly Ride Volume: Uber vs Lyft")
ax.set_ylabel("Number of sampled rides")
ax.set_xlabel("")
plt.xticks(rotation=45)
plt.show()

display(monthly_pivot)

## EDA 3 — Trip distribution by day and hour

In [ ]:
hour_day = (
    hvfhv
    .groupby(["day_name", "hour"])
    .size()
    .reset_index(name="rides")
)

hour_day["day_name"] = pd.Categorical(hour_day["day_name"], categories=day_order, ordered=True)

pivot_hour_day = hour_day.pivot(index="hour", columns="day_name", values="rides").fillna(0)

ax = pivot_hour_day.plot(figsize=(14,6))
ax.set_title("hvfhv Uber/Lyft Trip Distribution by Day and Hour")
ax.set_ylabel("Number of rides")
ax.set_xlabel("Hour of day")
plt.show()

In [ ]:
yellow_hour_day = (
    yellow
    .groupby(["day_name", "hour"])
    .size()
    .reset_index(name="rides")
)

yellow_hour_day["day_name"] = pd.Categorical(yellow_hour_day["day_name"], categories=day_order, ordered=True)

yellow_pivot_hour_day = yellow_hour_day.pivot(index="hour", columns="day_name", values="rides").fillna(0)

ax = yellow_pivot_hour_day.plot(figsize=(14,6))
ax.set_title("Yellow Taxi Trip Distribution by Day and Hour")
ax.set_ylabel("Number of rides")
ax.set_xlabel("Hour of day")
plt.show()

## EDA 4 — Uber vs Lyft market share

In [ ]:
company_share_daily = daily_pivot.div(daily_pivot.sum(axis=1), axis=0) * 100

ax = company_share_daily.plot(figsize=(14,5))
ax.set_title("Daily Share of hvfhv Rides: Uber vs Lyft")
ax.set_ylabel("Share of rides (%)")
ax.set_xlabel("")
plt.show()

display(company_share_daily.describe())

## EDA 5 — Passenger behavior / trip behavior

In [ ]:
## Distance buckets
def add_distance_bucket(df):
    df["distance_bucket"] = pd.cut(
        df["distance_miles"],
        bins=[0, 2, 6, 10, 20, np.inf],
        labels=["<2 miles", "2-6 miles", "6-10 miles", "10-20 miles", "20+ miles"],
        right=False
    )
    return df

yellow = add_distance_bucket(yellow)
hvfhv = add_distance_bucket(hvfhv)

def add_distance_bucket(df):
    df["distance_bucket"] = pd.cut(
        df["distance_miles"],
        bins=[0, 2, 6, 10, 20, np.inf],
        labels=["<2 miles", "2-6 miles", "6-10 miles", "10-20 miles", "20+ miles"],
        right=False
    )
    return df

yellow = add_distance_bucket(yellow)
hvfhv = add_distance_bucket(hvfhv)

In [ ]:
## Duration buckets
def add_duration_bucket(df):
    df["duration_bucket"] = pd.cut(
        df["trip_duration_min"],
        bins=[0, 5, 15, 30, 60, np.inf],
        labels=["<5 min", "5-15 min", "15-30 min", "30-60 min", "60+ min"],
        right=False
    )
    return df

yellow = add_duration_bucket(yellow)
hvfhv = add_duration_bucket(hvfhv)

plot_share(hvfhv["duration_bucket"], "hvfhv Uber/Lyft Trip Share by Duration")
plot_share(yellow["duration_bucket"], "Yellow Taxi Trip Share by Duration")

In [ ]:
## Passenger count distribution
plot_share(yellow["passenger_count"], "Yellow Taxi Passenger Count Distribution")